In [4]:
import pandas as pd
import numpy as np
import os

input_file = "pv_data.xlsx"
output_file = "pv_final_datasheet.xlsx"

if not os.path.exists(input_file):
    if os.path.exists("pv_data.xlsx - Combinations.csv"):
        input_file = "pv_data.xlsx - Combinations.csv"
    else:
        raise FileNotFoundError("Please ensure your full excel sheet is named 'pv_data.xlsx' in this folder.")

df = pd.read_csv(input_file) if input_file.endswith('.csv') else pd.read_excel(input_file)

df.columns = df.columns.astype(str).str.strip()
required_cols = ['G1', 'G2', 'G3', 'G4']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise KeyError(
        f"Required columns missing from input file: {missing_cols}. "
        f"Available columns: {list(df.columns)}"
    )

k, q = 1.38e-23, 1.60e-19
Isc, Voc, Ns = 8, 38, 54            
Ki, Kv, n = 0.0032, -0.123, 1.3            
Rsh, Rs, Ion = 415.405, 0.221, 9.825e-8    
T = 25 + 273.15
Vt = (Ns * k * T) / q 

def solve_module_voltage(I, G):
    I_pv = (Isc + Ki * (T - 298.15)) * (G / 1000)
    I_o_adj = Ion * ((T / 298.15)**3) * np.exp(((q * 1.12) / (n * k)) * (1/298.15 - 1/T))
    V = np.zeros_like(I)
    for i in range(len(I)):
        if I[i] > I_pv + I_o_adj: 
            V[i] = 0
            continue
        v_guess = 0.0
        for _ in range(5): 
            arg = (I_pv + I_o_adj - I[i] - v_guess / Rsh) / I_o_adj
            if arg <= 0:
                v_guess = 0
                break
            v_guess = n * Vt * np.log(arg) - I[i] * Rs
        V[i] = max(0, v_guess) 
    return V

def get_4S_mpp(g1, g2, g3, g4):
    I_mesh = np.linspace(0, Isc, 350)
    V = solve_module_voltage(I_mesh, g1) + solve_module_voltage(I_mesh, g2) + solve_module_voltage(I_mesh, g3) + solve_module_voltage(I_mesh, g4)
    P = V * I_mesh
    idx_max = np.argmax(P)
    return P[idx_max], V[idx_max], I_mesh[idx_max]

def get_4P_mpp(g1, g2, g3, g4):
    I_mesh = np.linspace(0, Isc, 350)
    v1, v2, v3, v4 = solve_module_voltage(I_mesh, g1), solve_module_voltage(I_mesh, g2), solve_module_voltage(I_mesh, g3), solve_module_voltage(I_mesh, g4)
    v_target = np.linspace(0, max(np.max(v1), np.max(v2), np.max(v3), np.max(v4)), 300)
    i1 = np.interp(v_target, v1[::-1], I_mesh[::-1], left=Isc, right=0)
    i2 = np.interp(v_target, v2[::-1], I_mesh[::-1], left=Isc, right=0)
    i3 = np.interp(v_target, v3[::-1], I_mesh[::-1], left=Isc, right=0)
    i4 = np.interp(v_target, v4[::-1], I_mesh[::-1], left=Isc, right=0)
    I_total = i1 + i2 + i3 + i4
    P = v_target * I_total
    idx_max = np.argmax(P)
    return P[idx_max], v_target[idx_max], I_total[idx_max]

def get_2S2P_mpp(g1, g2, g3, g4):
    I_mesh = np.linspace(0, Isc, 350)
    V_strA = solve_module_voltage(I_mesh, g1) + solve_module_voltage(I_mesh, g2)
    V_strB = solve_module_voltage(I_mesh, g3) + solve_module_voltage(I_mesh, g4)
    v_target = np.linspace(0, max(np.max(V_strA), np.max(V_strB)), 300)
    iA = np.interp(v_target, V_strA[::-1], I_mesh[::-1], left=Isc, right=0)
    iB = np.interp(v_target, V_strB[::-1], I_mesh[::-1], left=Isc, right=0)
    I_total = iA + iB
    P = v_target * I_total
    idx_max = np.argmax(P)
    return P[idx_max], v_target[idx_max], I_total[idx_max]

def get_2P2S_mpp(g1, g2, g3, g4):
    I_mesh = np.linspace(0, Isc, 350)
    v1, v2, v3, v4 = solve_module_voltage(I_mesh, g1), solve_module_voltage(I_mesh, g2), solve_module_voltage(I_mesh, g3), solve_module_voltage(I_mesh, g4)
    v_target_shared = np.linspace(0, max(np.max(v1), np.max(v2), np.max(v3), np.max(v4)), 200)
    i1 = np.interp(v_target_shared, v1[::-1], I_mesh[::-1], left=Isc, right=0)
    i2 = np.interp(v_target_shared, v2[::-1], I_mesh[::-1], left=Isc, right=0)
    i3 = np.interp(v_target_shared, v3[::-1], I_mesh[::-1], left=Isc, right=0)
    i4 = np.interp(v_target_shared, v4[::-1], I_mesh[::-1], left=Isc, right=0)
    
    I_series_string = np.linspace(0,  Isc, 350)
    V_blockA = np.interp(I_series_string, (i1 + i2)[::-1], v_target_shared[::-1], left=0, right=0)
    V_blockB = np.interp(I_series_string, (i3 + i4)[::-1], v_target_shared[::-1], left=0, right=0)
    V_total = V_blockA + V_blockB
    P = V_total * I_series_string
    idx_max = np.argmax(P)
    return P[idx_max], V_total[idx_max], I_series_string[idx_max]


# Lists for 4P
p_4p_list, v_4p_list, i_4p_list = [], [], []
# Lists for 4S
p_4s_list, v_4s_list, i_4s_list = [], [], []
# Lists for 2S2P
p_2s2p_list, v_2s2p_list, i_2s2p_list = [], [], []
# Lists for 2P2S
p_2p2s_list, v_2p2s_list, i_2p2s_list = [], [], []
# Overall lists
p_ovr_list, v_ovr_list, i_ovr_list = [], [], []

for idx, row in df.iterrows():
    g1, g2, g3, g4 = row['G1'], row['G2'], row['G3'], row['G4']
    
    p_4p, v_4p, i_4p = get_4P_mpp(g1, g2, g3, g4)
    p_4s, v_4s, i_4s = get_4S_mpp(g1, g2, g3, g4)
    p_2s2p, v_2s2p, i_2s2p = get_2S2P_mpp(g1, g2, g3, g4)
    p_2p2s, v_2p2s, i_2p2s = get_2P2S_mpp(g1, g2, g3, g4)
    
    # Append values (rounded to 2 decimal places)
    p_4p_list.append(round(p_4p, 2)); v_4p_list.append(round(v_4p, 2)); i_4p_list.append(round(i_4p, 2))
    p_4s_list.append(round(p_4s, 2)); v_4s_list.append(round(v_4s, 2)); i_4s_list.append(round(i_4s, 2))
    p_2s2p_list.append(round(p_2s2p, 2)); v_2s2p_list.append(round(v_2s2p, 2)); i_2s2p_list.append(round(i_2s2p, 2))
    p_2p2s_list.append(round(p_2p2s, 2)); v_2p2s_list.append(round(v_2p2s, 2)); i_2p2s_list.append(round(i_2p2s, 2))
    
    # Determine overall maximum power and its corresponding voltage and current
    powers = [p_4p, p_4s, p_2s2p, p_2p2s]
    voltages = [v_4p, v_4s, v_2s2p, v_2p2s]
    currents = [i_4p, i_4s, i_2s2p, i_2p2s]
    
    max_idx = np.argmax(powers)
    p_ovr_list.append(round(powers[max_idx], 2))
    v_ovr_list.append(round(voltages[max_idx], 2))
    i_ovr_list.append(round(currents[max_idx], 2))
    
    if (idx + 1) % 1000 == 0:
        print(f"Calculated: {idx + 1} / 10,000 rows...")

df['4P_mpp'] = p_4p_list
df['4P_Vmpp'] = v_4p_list
df['4P_Impp'] = i_4p_list

df['4S_mpp'] = p_4s_list
df['4S_Vmpp'] = v_4s_list
df['4S_Impp'] = i_4s_list

df['2S2P_mpp'] = p_2s2p_list
df['2S2P_Vmpp'] = v_2s2p_list
df['2S2P_Impp'] = i_2s2p_list

df['2P2S_mpp'] = p_2p2s_list
df['2P2S_Vmpp'] = v_2p2s_list
df['2P2S_Impp'] = i_2p2s_list

df['P_mpp_ovr'] = p_ovr_list
df['V_mpp_ovr'] = v_ovr_list
df['I_mpp_ovr'] = i_ovr_list

df.to_excel(output_file, index=False)
print(f"\nCompleted : '{output_file}'")


Calculated: 1000 / 10,000 rows...
Calculated: 2000 / 10,000 rows...
Calculated: 3000 / 10,000 rows...
Calculated: 4000 / 10,000 rows...
Calculated: 5000 / 10,000 rows...
Calculated: 6000 / 10,000 rows...
Calculated: 7000 / 10,000 rows...
Calculated: 8000 / 10,000 rows...
Calculated: 9000 / 10,000 rows...
Calculated: 10000 / 10,000 rows...

Completed : 'pv_final_datasheet.xlsx'
